In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    sha2,
    concat_ws,
    to_date
)

import uuid
import math

dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("chunk_column", "")
dbutils.widgets.text("start_id", "")
dbutils.widgets.text("end_id", "")

dbutils.widgets.text("jdbc_url", "")
dbutils.widgets.text("user", "")
dbutils.widgets.text("password", "")
dbutils.widgets.text("driver", "")
dbutils.widgets.text("source_name", "")

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
chunk_column = dbutils.widgets.get("chunk_column")

jdbc_url=dbutils.widgets.get("jdbc_url")
user=dbutils.widgets.get("user")
password=dbutils.widgets.get("password")
driver=dbutils.widgets.get("driver")

start_id = int(float(dbutils.widgets.get("start_id")))
end_id = int(float(dbutils.widgets.get("end_id")))
source_name = dbutils.widgets.get("source_name")
print("========== INPUTS ==========")
print("source_table:", source_table)
print("target_table:", target_table)
print("user:", user)
print("password:", password)
print("driver:", driver)
print("jdbc_url:", jdbc_url)

properties = {
    "user": user,
    "password": password,
    "driver": driver
}

query = f"""
(
    SELECT *
    FROM {source_table}
    WHERE {chunk_column}
    BETWEEN {start_id} AND {end_id}
) t
"""

df = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=properties
)
# =========================================================
# METADATA GENERATION
# =========================================================

batch_id = f"{start_id}_{end_id}"

pipeline_run_id = str(uuid.uuid4())

# Keep only original business columns for hash generation
business_columns = df.columns

df = (
    df.withColumn("migrated_at", current_timestamp())
      .withColumn("ingestion_date", to_date(current_timestamp()))
      .withColumn("batch_id", lit(batch_id))
      .withColumn("source_table", lit(source_table))
      .withColumn("source_system", lit(source_name))
      .withColumn("load_start_id", lit(start_id))
      .withColumn("load_end_id", lit(end_id))
      .withColumn("pipeline_run_id", lit(pipeline_run_id))
      .withColumn(
          "record_hash",
          sha2(concat_ws("||", *business_columns), 256)
      )
)

df.write.format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print("MANUAL CHUNK LOAD SUCCESS")
